# 数据预处理

读取 `数据/` 中的原始文件，按 ETF、指数及商品价格、宏观指标分别整理成长表；保留所有版本及来源，暂不筛选标的、计算收益、复权、填补缺失或拼接。请从 `coding` 目录运行。


## 1. 准备目录与读取函数

使用 Python 标准库读取 CSV 和 Excel 的原始缓存值。输出仅包含非空观测，空缺不填零；原始文件保持不变。

In [ ]:
from pathlib import Path
from collections import defaultdict
from datetime import datetime,timedelta
import csv,zipfile,xml.etree.ElementTree as ET,re
N={'m':'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
out=Path('整理数据'); out.mkdir(exist_ok=True)
cols=['日期','资产类别','代码','名称','字段','数值_原始','来源文件','来源工作表','来源列']
files={k:(out/(k+'.csv')).open('w',encoding='utf-8-sig',newline='') for k in ['ETF汇总','指数及商品价格汇总','宏观指标汇总']}
writers={k:csv.writer(v) for k,v in files.items()}
for w in writers.values():w.writerow(cols)
stats=defaultdict(lambda:[0,None,None,0]); duplicates=defaultdict(int)
def emit(group,date,category,code,name,field,value,source,sheet,col):
 if value is None or str(value).strip()=='':return
 date=str(date)[:10]; value=str(value).strip(); key=(group,category,code,name,field,str(source),sheet,col)
 writers[group].writerow([date,category,code,name,field,value,str(source),sheet,col]); s=stats[key];s[0]+=1;s[1]=min(s[1] or date,date);s[2]=max(s[2] or date,date)
 try:float(value)
 except ValueError:s[3]+=1
def dateval(v):
 try:return (datetime(1899,12,30)+timedelta(days=float(v))).strftime('%Y-%m-%d')
 except (ValueError,TypeError,OverflowError):
  x=str(v or '')[:10];return x if re.fullmatch(r'\d{4}-\d\d-\d\d',x) else None
def colnum(ref):
 n=0
 for c in re.match(r'[A-Z]+',ref).group():n=n*26+ord(c)-64
 return n
def read_xlsx(p):
 with zipfile.ZipFile(p) as z:
  names=z.namelist(); shared=[]
  if 'xl/sharedStrings.xml' in names:
   shared=[''.join(t.text or '' for t in x.findall('.//m:t',N)) for x in ET.fromstring(z.read('xl/sharedStrings.xml')).findall('m:si',N)]
  wb=ET.fromstring(z.read('xl/workbook.xml')); sheetnames=[x.get('name') for x in wb.findall('.//m:sheet',N)]
  sheets=sorted((n for n in names if re.fullmatch(r'xl/worksheets/sheet\d+\.xml',n)),key=lambda x:int(re.search(r'sheet(\d+)',x).group(1)))
  for i,f in enumerate(sheets):
   root=ET.fromstring(z.read(f)); rows={}
   for row in root.findall('.//m:sheetData/m:row',N):
    d={}
    for c in row.findall('m:c',N):
     ref=c.get('r');v=c.find('m:v',N); val=v.text if v is not None else ''.join(t.text or '' for t in c.findall('.//m:t',N))
     if c.get('t')=='s' and val:val=shared[int(val)]
     d[colnum(ref)]=val
    rows[int(row.get('r'))]=d
   yield sheetnames[i] if i<len(sheetnames) else f,rows


## 2. 整合各来源

保留原始字段和来源文件；商品期货价格作为指数版中的价格代理单独标识，不视为可交易收益。

In [ ]:
for p in sorted(Path('数据').rglob('*')):
 if not p.is_file():continue
 source=p.as_posix(); category=p.parent.name
 if p.suffix.lower()=='.csv':
  raw=p.read_bytes()
  for enc in ('utf-8-sig','gb18030'):
   try:content=raw.decode(enc);break
   except UnicodeError:continue
  rows=list(csv.reader(content.splitlines())); header=rows[0]
  if header[0]=='Wind代码':
   isetf='ETF' in p.name; group='ETF汇总' if isetf else '指数及商品价格汇总'
   for row in rows[1:]:
    if len(row)<6:continue
    date=dateval(row[2])
    if not date:continue
    for c in (3,5):emit(group,date,category,row[0],row[1],header[c],row[c],source,'CSV',str(c+1))
  else:
   macro=p.name in ('宏观指标.csv','领先指标.csv');group='宏观指标汇总' if macro else '指数及商品价格汇总'
   for row in rows[1:]:
    date=dateval(row[0]) if row else None
    if not date:continue
    for c in range(1,min(len(row),len(header))):emit(group,date,'宏观' if macro else category,'',header[c],'原始指标值' if macro else '原始指数或价格',row[c],source,'CSV',str(c+1))
 elif p.suffix.lower()=='.xlsx':
  for sheet,rows in read_xlsx(p):
   if '中债-国债总' in p.name:
    title=rows.get(1,{}).get(1,''); names=rows.get(3,{});codes=rows.get(4,{})
    for rn,row in rows.items():
     if rn<5:continue
     date=dateval(row.get(1))
     if date:
      for c,name in names.items():
       if c>1:emit('指数及商品价格汇总',date,category,codes.get(c,''),title+' '+name,'指数点位',row.get(c),source,sheet,str(c))
   else:
    heads=rows.get(3,{});names=rows.get(4,{});codes=rows.get(5,{}); starts=sorted(c for c,v in names.items() if v=='日期')
    group='指数及商品价格汇总' if p.name=='商品.xlsx' else 'ETF汇总'
    for j,start in enumerate(starts):
     end=starts[j+1] if j+1<len(starts) else max(codes.keys(),default=start)+1
     field=heads.get(start,'未知字段')+' ['+str(heads.get(start+1,'未标识'))+']'
     for rn,row in rows.items():
      if rn<6:continue
      date=dateval(row.get(start))
      if not date:continue
      for c in range(start+1,end):
       if codes.get(c) or names.get(c):emit(group,date,category,codes.get(c,''),names.get(c,''),field,row.get(c),source,sheet,str(c))
for f in files.values():f.close()
with (out/'数据清单.csv').open('w',encoding='utf-8-sig',newline='') as f:
 w=csv.writer(f);w.writerow(['汇总版本','资产类别','代码','名称','字段','来源文件','来源工作表','来源列','非空记录数','首个日期','最后日期','非数值记录数'])
 for k,v in sorted(stats.items()):w.writerow([*k,*v])
print('counts', {g:sum(v[0] for k,v in stats.items() if k[0]==g) for g in files},'series',len(stats))
print('nonnumeric',[(k,v[3]) for k,v in stats.items() if v[3]])


## 3. 检查输出

检查数据清单、复权口径、重复版本和可用区间后，再决定模型资产池。

In [ ]:
from pathlib import Path
for p in sorted(Path('整理数据').glob('*.csv')):
    print(f'{p.name}: {p.stat().st_size:,} bytes')
